# LLM API Calling & Learning

This notebook walks through working with LLM APIs step-by-step.

## What You'll Learn
- How to make your first LLM API call (Google Gemini + OpenRouter)
- How to use LangChain as an abstraction layer
- How to build a simple agent with tools

## Prerequisites
1. Copy `.env.example` to `.env` and add your API keys
2. Run: `pip install -r requirements.txt`
3. Each section can be run independently

## Important Notes
- Run cells **in order** — later cells depend on variables defined earlier
- API calls cost tokens — free tier limits apply
- See `docs/PROVIDER_SETUP.md` for API key setup

---
## Section 1: Setup and Imports

Load your API keys from `.env` and import the required libraries.

In [ ]:
import os
import sys
from dotenv import load_dotenv

from langchain_openai import ChatOpenAI
from google import genai

# Load keys from .env file
load_dotenv()

# Read keys — will raise KeyError with a helpful message if missing
openrouter_key = os.getenv('OPENROUTER_API_KEY')
gemini_key = os.getenv('GEMINI_API_KEY')

if not openrouter_key:
    print("WARNING: OPENROUTER_API_KEY not set. OpenRouter examples will not work.")
    print("  Get a free key at: https://openrouter.ai/keys")

if not gemini_key:
    print("WARNING: GEMINI_API_KEY not set. Gemini examples will not work.")
    print("  Get a free key at: https://aistudio.google.com/apikey")

print("Setup complete.")

---
## Section 2: LangChain with OpenRouter

**What is LangChain?** A framework that wraps multiple LLM providers behind a common interface.

**What is OpenRouter?** A service that gives you access to many AI models (including free ones) via a single API key.

> ⚠️ **Important:** Free models on OpenRouter change frequently. Check current free models at: https://openrouter.ai/models?q=:free

The `ChatOpenAI` class from LangChain can be pointed at OpenRouter by setting `base_url`.

In [ ]:
# Initialize LangChain's ChatOpenAI pointed at OpenRouter
# Replace the model name with a current free model from https://openrouter.ai/models?q=:free
OPENROUTER_FREE_MODEL = "google/gemma-4-31b-it:free"  # Verified free as of 2026-10-30

if openrouter_key:
    llm_openrouter = ChatOpenAI(
        model=OPENROUTER_FREE_MODEL,
        base_url="https://openrouter.ai/api/v1",
        api_key=openrouter_key,
    )
    print(f"OpenRouter client ready. Model: {OPENROUTER_FREE_MODEL}")
else:
    print("Skipping OpenRouter setup — key not available.")
    llm_openrouter = None

In [ ]:
# Make a request using LangChain's invoke() method
# invoke() returns a message object; use .content to get the text
if llm_openrouter:
    response_openrouter = llm_openrouter.invoke("What is Python? Answer in 3 sentences.")
    print("=== OpenRouter Response ===")
    print(response_openrouter.content)
else:
    print("No OpenRouter key — skipping.")

---
## Section 3: Google Gemini (direct SDK)

Here we use the `google-genai` SDK directly (not through LangChain).

**Note:** The correct model ID as of 2026-10-30 is `gemini-3.8-flash`.
Always verify current model IDs at: https://ai.google.dev/gemini-api/docs/models

In [ ]:
# Initialize the Gemini client
GEMINI_MODEL = os.getenv("GEMINI_MODEL", "gemini-3.8-flash")  # Verified 2026-10-30

if gemini_key:
    gemini_client = genai.Client(api_key=gemini_key)
    print(f"Gemini client ready. Model: {GEMINI_MODEL}")
else:
    print("Skipping Gemini setup — key not available.")
    gemini_client = None

In [ ]:
# Make a request to Gemini
# generate_content() takes the model name and content (your prompt)
# response.text gives you the plain text output
if gemini_client:
    response_gemini = gemini_client.models.generate_content(
        model=GEMINI_MODEL,
        contents="What is Python? Answer in 3 sentences."
    )
    print("=== Gemini Response ===")
    print(response_gemini.text)
else:
    print("No Gemini key — skipping.")

---
## Section 4: Comparing Responses

When you have both keys, you can compare how different models respond to the same prompt.

**Notice:** The same question gets different answers! This is because:
- Different training data
- Different model sizes and capabilities
- Random temperature parameter

In [ ]:
COMPARISON_PROMPT = "Explain what an API is in one sentence."

if llm_openrouter:
    resp1 = llm_openrouter.invoke(COMPARISON_PROMPT)
    print(f"OpenRouter ({OPENROUTER_FREE_MODEL}):")
    print(f"  {resp1.content}\n")

if gemini_client:
    resp2 = gemini_client.models.generate_content(
        model=GEMINI_MODEL,
        contents=COMPARISON_PROMPT
    )
    print(f"Gemini ({GEMINI_MODEL}):")
    print(f"  {resp2.text}")

---
## Section 5: Building a Simple Agent

An **agent** is an LLM that can use tools to answer questions.

Here we build an agent that can:
- Look up the current time in a city
- Perform simple calculations

**Key concept:** The LLM doesn't actually execute the tools — it *requests* tool calls, and our Python code executes them.

> ⚠️ **Note:** This section requires LangChain >= 0.3. See Module 10 for a more detailed agent implementation.

In [ ]:
from datetime import datetime
from zoneinfo import ZoneInfo
from langchain_core.tools import tool

@tool
def get_city_time(city: str) -> str:
    """Get the current local time for a supported city.
    
    Args:
        city: The city name (supported: india, london, new_york, tokyo, dubai)
    """
    # A simple lookup — in production, use a geolocation service
    TIMEZONES = {
        "india": "Asia/Kolkata",
        "london": "Europe/London",
        "new york": "America/New_York",
        "new_york": "America/New_York",
        "tokyo": "Asia/Tokyo",
        "dubai": "Asia/Dubai",
    }
    
    timezone_name = TIMEZONES.get(city.lower())
    if timezone_name is None:
        supported = ", ".join(TIMEZONES.keys())
        return f"City '{city}' not supported. Supported cities: {supported}"
    
    current_time = datetime.now(ZoneInfo(timezone_name))
    return current_time.strftime("%Y-%m-%d %I:%M:%S %p %Z")


@tool
def calculate(expression: str) -> str:
    """Evaluate a simple arithmetic expression safely.
    
    Args:
        expression: A simple math expression like '2 + 2' or '10 * 5'
    """
    import ast
    import operator
    
    # SAFE evaluation: only allow basic arithmetic, not arbitrary code
    ALLOWED_OPERATORS = {
        ast.Add: operator.add,
        ast.Sub: operator.sub,
        ast.Mult: operator.mul,
        ast.Div: operator.truediv,
        ast.Pow: operator.pow,
    }
    
    def safe_eval(node):
        if isinstance(node, ast.Constant):
            return node.value
        elif isinstance(node, ast.BinOp):
            op_type = type(node.op)
            if op_type not in ALLOWED_OPERATORS:
                raise ValueError(f"Operator not allowed: {op_type}")
            return ALLOWED_OPERATORS[op_type](safe_eval(node.left), safe_eval(node.right))
        else:
            raise ValueError(f"Unsupported expression element: {type(node)}")
    
    try:
        tree = ast.parse(expression.strip(), mode='eval')
        result = safe_eval(tree.body)
        return str(result)
    except Exception as e:
        return f"Error: {e}. Only basic arithmetic is supported."


print("Tools defined:")
print(f"  - {get_city_time.name}: {get_city_time.description}")
print(f"  - {calculate.name}: {calculate.description}")

In [ ]:
# Create an agent using LangGraph (modern LangChain approach)
# Note: create_react_agent is from langgraph, not langchain.agents
from langgraph.prebuilt import create_react_agent

TOOLS = [get_city_time, calculate]

if llm_openrouter:
    # Bind tools to the LLM — lets the model request tool calls
    agent = create_react_agent(
        model=llm_openrouter,
        tools=TOOLS,
    )
    print("Agent created with OpenRouter model.")
elif gemini_client:
    # For Gemini via LangChain
    from langchain_google_genai import ChatGoogleGenerativeAI
    llm_gemini = ChatGoogleGenerativeAI(model=GEMINI_MODEL, google_api_key=gemini_key)
    agent = create_react_agent(model=llm_gemini, tools=TOOLS)
    print("Agent created with Gemini model.")
else:
    agent = None
    print("No LLM configured. Skipping agent creation.")

In [ ]:
# Run the agent with a question that requires tool use
# The agent will:
# 1. Reason about what tool to call
# 2. Call the tool
# 3. Observe the result
# 4. Produce a final answer

if agent:
    result = agent.invoke({
        "messages": [{"role": "user", "content": "What is the current time in London?"}]
    })
    
    # The last message in the list is the agent's final answer
    final_message = result["messages"][-1]
    print("=== Agent Response ===")
    print(final_message.content)
else:
    print("Agent not available — no API key configured.")

In [ ]:
# Test the agent's calculator tool
if agent:
    result = agent.invoke({
        "messages": [{"role": "user", "content": "What is 15 * 8 + 100?"}]
    })
    print("=== Calculator Agent ===")
    print(result["messages"][-1].content)

---
## Exercises

Try these to deepen your understanding:

1. **Change the prompt**: Ask the agent about time in a city that isn't in the lookup (e.g., 'Paris'). What happens?

2. **Add a tool**: Create a new `@tool` function that returns the day of the week for a given date. Register it with the agent.

3. **Compare models**: If you have both API keys, create two agents with different models. Ask both the same question. Compare their reasoning steps.

4. **Error handling**: What happens if you give the calculator a division by zero? Improve the `calculate` tool to handle this gracefully.

5. **System prompt**: Modify `create_react_agent` to include a system prompt that constrains the agent's behavior. See the LangGraph docs for the `state_modifier` parameter.

---
## Next Steps

- **Module 10** (`10_agents/`): A deeper dive into agents — state, stop conditions, observability
- **Module 07** (`07_langchain_fundamentals/`): LangChain patterns — prompt templates, runnables, output parsers
- **Project 09** (`projects/09_chat_with_pdf/`): Build a document Q&A system

**References:**
- [LangGraph docs](https://langchain-ai.github.io/langgraph/)
- [google-genai SDK](https://github.com/googleapis/python-genai)
- [OpenRouter model list](https://openrouter.ai/models)